# Test Full Pipeline v1.3 — via `src` package

This notebook tests the **full retrieval pipeline v1.3** by importing the
extracted, reusable functions from the `src/` package (instead of redefining
everything inline).

Stages: build pipeline → smoke test → manual demo → evaluation.

> **Note on device:** if local CUDA is broken (Windows driver mismatch), set
> `VNLEGAL_DEVICE=cpu` *before* importing `src` (cell below).

## 1. Setup — put repo root on `sys.path`

In [1]:
import os
import sys
from pathlib import Path

# Force CPU here if your local CUDA runtime is broken; remove to auto-detect.
os.environ.setdefault("VNLEGAL_DEVICE", "cpu")

# Locate repo root (parent of this notebook's pipeline_v1.3 dir) and add to path.
repo_root = Path.cwd()
while not (repo_root / "src").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("repo_root:", repo_root)

repo_root: C:\Users\hung\Documents\GitHub\vnlegal-rag


## 2. Build the pipeline

Resolves data + artifacts, loads TextCNN, builds the TF-IDF index, and (if Siamese weights exist) precomputes document embeddings.

In [2]:
from src import PipelineConfig, RetrievalPipeline, build_markdown_table

config = PipelineConfig()
print("device:", config.device)

pipe = RetrievalPipeline.from_config(config)
print("Siamese available:", pipe.has_siamese)

device: cpu


[data] dir=data_ready_v1_3_k3  corpus=9715  qa_test=2991  labels=6
[textcnn] textcnn_artifacts_v1_3  max_len=256


[tfidf] matrix=(9715, 6033)
[siamese] absent — Siamese modes disabled
Siamese available: False


## 3. Smoke test — one query through every available mode

In [3]:
from src.retrieval import predict_topic_topk

question = "Hợp đồng lao động thời vụ có bắt buộc đóng bảo hiểm xã hội không?"

# Predicted macro_domain (topic) labels from TextCNN
ids, labels, probs = predict_topic_topk(question, pipe.textcnn, pipe.config.device, k=3)
print("Top topics:", list(zip(labels, [round(float(p), 3) for p in probs])))

print("\n[TF-IDF + TextCNN] top-5")
display(pipe.retrieve_tfidf_textcnn(question, k=5))

if pipe.has_siamese:
    print("\n[Siamese + TextCNN] top-5")
    display(pipe.retrieve_siamese_textcnn(question, k=5))
    print("\n[Siamese only] top-5")
    display(pipe.retrieve_siamese_only(question, k=5))
else:
    print("\n[Siamese modes skipped — no Siamese weights found]")

Top topics: [('Labor & Insurance', 0.971), ('other', 0.008), ('Transportation', 0.008)]

[TF-IDF + TextCNN] top-5


,score,passage_id,macro_domain,article_content
0,0.790364,du_thao_luat_bao_hiem_xa_hoi_dieu_36_71b61966,Labor & Insurance,"Điều 36. Chậm đóng, trốn đóng bảo hiểm xã hội ..."
1,0.704657,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 13. Trách nhiệm của người sử dụng lao độn...
2,0.689870,du_thao_luat_bao_hiem_xa_hoi_dieu_110_d869b78c,Labor & Insurance,Điều 110. Chế độ hưu trí và chế độ tử tuất đối...
3,0.679229,du_thao_luat_bao_hiem_xa_hoi_dieu_12_6228f499,Labor & Insurance,Điều 12. Trách nhiệm của người sử dụng lao độn...
4,0.669796,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 111. Chế độ hưu trí và chế độ tử tuất đối...



[Siamese modes skipped — no Siamese weights found]


## 4. Manual demo — try your own query

In [4]:
my_query = "Quy định về thời gian nghỉ thai sản của lao động nữ?"

_, my_labels, _ = predict_topic_topk(my_query, pipe.textcnn, pipe.config.device, k=3)
print("Predicted topics:", my_labels)
display(pipe.retrieve_tfidf_textcnn(my_query, k=5))

Predicted topics: ['Labor & Insurance', 'Finance & Banking', 'other']


,score,passage_id,macro_domain,article_content
0,0.656850,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 54. Chế độ thai sản của lao động nữ mang ...
1,0.644236,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 51. Thời gian nghỉ việc hưởng chế độ thai...
2,0.641463,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_58_2014_q...,Labor & Insurance,Điều 37. Thời gian hưởng chế độ khi thực hiện ...
3,0.633214,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 55. Chế độ thai sản của lao động nữ nhờ m...
4,0.631999,du_thao_luat_bao_hiem_xa_hoi_dieu_55_9e4288fa,Labor & Insurance,Điều 55. Thời gian hưởng chế độ khi thực hiện ...


## 5. Evaluation on `qa_test`

Recall@{1,3,5,10} and MRR over a sampled subset. Siamese rows appear only when weights are present.

In [5]:
pipe.config.eval_sample_n = 200   # set to None for the full split
results = pipe.evaluate("test")
print(build_markdown_table(results))

| pipeline      |   n_queries |   Recall@1 |   Recall@3 |   Recall@5 |   Recall@10 |    MRR |
|:--------------|------------:|-----------:|-----------:|-----------:|------------:|-------:|
| tfidf_textcnn |         200 |     0.5250 |     0.6950 |     0.7800 |      0.8300 | 0.6245 |


## 6. Notes

- All logic lives in `src/` — this notebook only orchestrates.
- Missing Siamese weights → those modes are skipped automatically.
- On Colab/Kaggle remove the `VNLEGAL_DEVICE=cpu` line to use the GPU.